# 07 - Recommendation and Dataset Testing

Notebook ini digunakan untuk menguji dataset, similarity matrix, fungsi rekomendasi, randomisasi hasil, dan filter kategori. Tujuannya adalah memastikan pipeline siap untuk tahap persiapan itinerary.

In [ ]:
import pickle
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/processed")
TFIDF_DATASET_FILE = DATA_DIR / "tfidf_dataset.csv"
SIMILARITY_MATRIX_FILE = DATA_DIR / "cosine_similarity_matrix.pkl"

OUTPUT_COLUMNS = ["name", "category", "rating", "type", "price_estimate", "similarity_score"]
VALID_TYPES = {"wisata", "kuliner", "hotel", "oleh_oleh"}

In [ ]:
# Load dataset dan similarity matrix.
df = pd.read_csv(TFIDF_DATASET_FILE)

with open(SIMILARITY_MATRIX_FILE, "rb") as file:
    cosine_sim_matrix = pickle.load(file)

print("Dataset shape:", df.shape)
print("Similarity matrix shape:", cosine_sim_matrix.shape)
print("Type counts:")
print(df["type"].value_counts())

In [ ]:
# Validasi dataset dasar.
required_columns = ["name", "category", "rating", "address", "subtypes", "type", "price_estimate", "combined_text"]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Kolom wajib belum tersedia: {missing_columns}")

validation_summary = pd.DataFrame({
    "metric": [
        "row_count",
        "duplicate_name_address",
        "null_count",
        "invalid_type_count",
        "invalid_rating_count",
        "invalid_price_count",
    ],
    "value": [
        len(df),
        int(df.duplicated(subset=["name", "address"]).sum()),
        int(df[required_columns].isna().sum().sum()),
        int((~df["type"].isin(VALID_TYPES)).sum()),
        int(((df["rating"] < 0) | (df["rating"] > 5)).sum()),
        int(((df["price_estimate"] < 10_000) | (df["price_estimate"] > 1_000_000)).sum()),
    ]
})

validation_summary

In [ ]:
def find_place_index(place_name, dataset):
    """Mencari index tempat berdasarkan nama."""
    keyword = str(place_name).lower().strip()
    matches = dataset[dataset["name"].str.lower().str.contains(keyword, na=False)]

    if matches.empty:
        raise ValueError(f"Tempat tidak ditemukan: {place_name}")

    return int(matches.index[0])


def recommend_similar_places(
    place_name,
    dataset,
    similarity_matrix,
    top_n=5,
    pool_size=20,
    filter_type=None,
    random_state=None
):
    """Fungsi testing rekomendasi dengan randomized recommendation pool."""
    if filter_type is not None and filter_type not in VALID_TYPES:
        raise ValueError(f"filter_type harus salah satu dari {sorted(VALID_TYPES)} atau None.")

    place_index = find_place_index(place_name, dataset)
    scores = similarity_matrix[place_index].copy()
    scores[place_index] = -1

    candidates = dataset.copy()
    candidates["similarity_score"] = scores

    if filter_type is not None:
        candidates = candidates[candidates["type"] == filter_type]

    candidates = candidates[candidates["similarity_score"] > 0]
    candidates = candidates.sort_values("similarity_score", ascending=False)
    top_pool = candidates.head(pool_size)

    if len(top_pool) <= top_n:
        selected = top_pool
    else:
        selected = top_pool.sample(n=top_n, random_state=random_state)
        selected = selected.sort_values("similarity_score", ascending=False)

    return selected[OUTPUT_COLUMNS].reset_index(drop=True)

In [ ]:
# Similarity testing: tampilkan tempat paling mirip untuk satu contoh wisata.
test_place = df[df["type"] == "wisata"].iloc[0]["name"]
print("Test place:", test_place)

similarity_test = recommend_similar_places(
    test_place,
    df,
    cosine_sim_matrix,
    top_n=5,
    pool_size=10,
    filter_type="wisata",
    random_state=1
)

similarity_test

In [ ]:
# Randomization testing: hasil dapat berbeda karena dipilih dari top similarity pool.
run_1 = recommend_similar_places(test_place, df, cosine_sim_matrix, top_n=5, pool_size=20, filter_type="wisata", random_state=10)
run_2 = recommend_similar_places(test_place, df, cosine_sim_matrix, top_n=5, pool_size=20, filter_type="wisata", random_state=25)

comparison = pd.DataFrame({
    "run_1": run_1["name"],
    "run_2": run_2["name"]
})

comparison

In [ ]:
# Recommendation diversity: jumlah item unik dari dua percobaan rekomendasi.
unique_recommendations = set(run_1["name"]).union(set(run_2["name"]))

diversity_summary = pd.DataFrame({
    "metric": ["run_1_count", "run_2_count", "unique_recommendation_count"],
    "value": [len(run_1), len(run_2), len(unique_recommendations)]
})

diversity_summary

In [ ]:
# Category filtering examples untuk empat kategori utama.
filter_examples = []

for category_type in ["wisata", "kuliner", "hotel", "oleh_oleh"]:
    place_name = df[df["type"] == category_type].iloc[0]["name"]
    recommendations = recommend_similar_places(
        place_name,
        df,
        cosine_sim_matrix,
        top_n=5,
        pool_size=20,
        filter_type=category_type,
        random_state=7
    )
    filter_examples.append({
        "category": category_type,
        "input_place": place_name,
        "recommendation_count": len(recommendations),
        "result_types": ", ".join(sorted(recommendations["type"].unique()))
    })

pd.DataFrame(filter_examples)

In [ ]:
# Sample output rekomendasi yang siap dipakai untuk persiapan itinerary.
oleh_oleh_place = "Krisna Oleh-Oleh Nusantara Jogja"
print("Input oleh-oleh:", oleh_oleh_place)

recommend_similar_places(
    oleh_oleh_place,
    df,
    cosine_sim_matrix,
    top_n=5,
    pool_size=20,
    filter_type="oleh_oleh",
    random_state=12
)

## Catatan Pengujian

Pengujian menunjukkan bahwa dataset sudah memiliki struktur yang dibutuhkan, similarity matrix sesuai dengan jumlah data, filter kategori dapat digunakan, dan randomized recommendation pool dapat menghasilkan variasi rekomendasi. Hasil rekomendasi tetap berbasis kemiripan konten dan belum mencakup optimasi rute, transportasi real-time, atau harga real-time.